#  HumanMM — Global Human Motion Recovery from Multi-Shot Videos
## Pipeline Evaluation Notebook

**Project:** Final Year Project  
**Pipeline:** YOLO11 → ByteTrack → MediaPipe → 3D Motion Recovery → Trajectory Alignment  
**Evaluation:** Per-frame metrics across all pipeline stages (treated as evaluation epochs)

---

This notebook:
1. Runs the full HumanMM pipeline on the input video
2. Records per-frame (epoch-wise) metrics for every stage
3. Plots detection confidence, pose confidence, FPS, person count over time
4. Shows detection quality histograms and performance breakdowns
5. Visualises sample outputs from each pipeline stage


## 1. Setup & Imports

In [1]:
import os, sys, time, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Make sure we're in the project root ──────────────────────────────────────
PROJECT_ROOT = Path('.').resolve()
# If notebook is in a subfolder, walk up to find main.py
while not (PROJECT_ROOT / 'main.py').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f'✅ Project root: {PROJECT_ROOT}')

# ── Core libs ────────────────────────────────────────────────────────────────
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')  # headless backend (works in Jupyter too)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from IPython.display import Image as IPImage, display, HTML

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'font.family':      'monospace',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})
ACCENT_GREEN  = '#3fb950'
ACCENT_BLUE   = '#58a6ff'
ACCENT_ORANGE = '#f0883e'
ACCENT_PURPLE = '#bc8cff'
ACCENT_RED    = '#ff7b72'
ACCENT_YELLOW = '#d29922'

print('✅ Imports complete')

✅ Project root: D:\Final_year
✅ Imports complete


## 2. Configuration

In [2]:
# ── Edit this cell if your video path is different ───────────────────────────
VIDEO_PATH  = 'data/input/demo.mp4'
OUTPUT_DIR  = 'outputs'
CONFIG_PATH = 'configs/config.yaml'
SAVE_FIGS   = True           # Save all figures to outputs/eval_figures/
FIG_DIR     = Path(OUTPUT_DIR) / 'eval_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Verify
assert Path(VIDEO_PATH).exists(),  f'❌ Video not found: {VIDEO_PATH}'
assert Path(CONFIG_PATH).exists(), f'❌ Config not found: {CONFIG_PATH}'
print(f'✅ Video  : {VIDEO_PATH}')
print(f'✅ Output : {OUTPUT_DIR}')
print(f'✅ Figures: {FIG_DIR}')

# ── Read video metadata ───────────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
VIDEO_FPS    = cap.get(cv2.CAP_PROP_FPS)
VIDEO_W      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_H      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
VIDEO_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VIDEO_DUR    = VIDEO_FRAMES / VIDEO_FPS
cap.release()

print()
print(f'📹 Video Properties')
print(f'   Resolution : {VIDEO_W}×{VIDEO_H}')
print(f'   Frame Rate : {VIDEO_FPS:.2f} fps')
print(f'   Total Frames: {VIDEO_FRAMES}')
print(f'   Duration   : {VIDEO_DUR:.2f} s')

✅ Video  : data/input/demo.mp4
✅ Output : outputs
✅ Figures: outputs\eval_figures

📹 Video Properties
   Resolution : 1920×1080
   Frame Rate : 30.00 fps
   Total Frames: 354
   Duration   : 11.80 s


## 3. Load HumanMM Pipeline & Run Inference

> **Each frame = one evaluation step (epoch-equivalent)**  
> We record detection confidence, pose confidence, FPS, and person count per frame.

In [3]:
from utils.config_loader import load_config
from utils.logger import configure_logging, get_logger

configure_logging(log_dir=f'{OUTPUT_DIR}/logs', level='WARNING', colorize=False)
logger = get_logger('notebook')

cfg = load_config(CONFIG_PATH, overrides=[f'video.path={VIDEO_PATH}',
                                           f'output.root_dir={OUTPUT_DIR}'])
print('✅ Config loaded')
print(f'   Detector backend : YOLO11 / YOLOv8 (auto-select)')
print(f'   Tracker backend  : {cfg.get("tracker", {}).get("backend", "bytetrack")}')
print(f'   Pose backend     : {cfg.get("pose", {}).get("backend", "mediapipe")}')

2026-07-04 09:23:18 | INFO     | utils.logger:configure_logging:112 | Logging configured — level=INFO log_dir=outputs\logs
2026-07-04 09:23:19 | INFO     | utils.config_loader:load_config:91 | Configuration loaded successfully (12 top-level keys)


✅ Config loaded
   Detector backend : YOLO11 / YOLOv8 (auto-select)
   Tracker backend  : bytetrack
   Pose backend     : mediapipe


In [4]:
# ── Initialise models ─────────────────────────────────────────────────────────
from models.model_factory import ModelFactory

device = cfg.get('device', {}).get('backend', 'cpu') if isinstance(cfg, dict) else 'cpu'
factory = ModelFactory(cfg, device=device)

print('Loading YOLO detector...')
detector_model = factory.create_detector()

print('Loading ByteTrack tracker...')
tracker_model = factory.create_tracker()

print('Loading MediaPipe pose estimator...')
pose_model = factory.create_pose_estimator()

print('\n✅ All models loaded')

2026-07-04 09:23:38 | INFO     | models.model_factory:create_detector:83 | ModelFactory: creating YOLODetector (device=cpu)
2026-07-04 09:23:38 | INFO     | models.base_model:initialize:70 | Initializing model: YOLO11/YOLOv8x


Loading YOLO detector...


2026-07-04 09:24:01 | INFO     | models.yolo_detector:load:257 | Loading YOLO model: yolo11n.pt on cpu
2026-07-04 09:24:04 | INFO     | models.yolo_detector:load:278 | YOLO ready: yolo11n.pt (segmentation=False) | warm-up complete
2026-07-04 09:24:04 | INFO     | models.base_model:initialize:74 | Model loaded successfully: YOLO11/YOLOv8x
2026-07-04 09:24:04 | INFO     | models.model_factory:create_tracker:111 | ModelFactory: creating tracker backend=bytetrack
2026-07-04 09:24:04 | INFO     | models.base_model:initialize:70 | Initializing model: ByteTrack
2026-07-04 09:24:04 | INFO     | models.base_model:initialize:74 | Model loaded successfully: ByteTrack
2026-07-04 09:24:04 | INFO     | models.model_factory:create_pose_estimator:155 | ModelFactory: creating pose estimator backend=mediapipe
2026-07-04 09:24:04 | INFO     | models.base_model:initialize:70 | Initializing model: MediaPipePose


Loading ByteTrack tracker...
Loading MediaPipe pose estimator...


2026-07-04 09:24:11 | INFO     | models.base_model:initialize:74 | Model loaded successfully: MediaPipePose



✅ All models loaded


In [5]:
# ── Stage wrappers ────────────────────────────────────────────────────────────
from pipeline.human_detector import HumanDetector
from pipeline.tracker import PersonTracker
from pipeline.pose_estimator import PoseEstimator
from pipeline.video_loader import VideoLoader

human_detector = HumanDetector(model=detector_model, max_persons=10)
person_tracker = PersonTracker(model=tracker_model)
pose_estimator = PoseEstimator(model=pose_model)

print('✅ Pipeline stages ready')

✅ Pipeline stages ready


In [6]:
# ── Load frames ───────────────────────────────────────────────────────────────
print('Loading video frames...')
loader = VideoLoader(VIDEO_PATH, config=cfg.get('video', {}) if isinstance(cfg, dict) else {})
loader.load()
frames = loader.read_all_frames()
print(f'✅ Loaded {len(frames)} frames')

2026-07-04 09:24:33 | INFO     | pipeline.video_loader:load:169 | Video loaded: demo.mp4 | 1920×1080 | 30.00 fps | 354 frames | 11.8s | 4.7 MB


Loading video frames...


2026-07-04 09:24:40 | INFO     | pipeline.video_loader:frame_iterator:264 | Frame extraction complete: 354 frames | 6.2s | 57.3 fps


✅ Loaded 354 frames


In [7]:
# ── Per-frame (epoch) inference + metrics recording ───────────────────────────
print('Running inference... (this is the evaluation loop — each frame = 1 epoch step)\n')

# Metrics storage
epoch_metrics = {
    'frame_idx':       [],
    'timestamp_sec':   [],

    # Detection metrics
    'det_count':       [],
    'det_conf_mean':   [],
    'det_conf_max':    [],
    'det_conf_min':    [],
    'det_latency_ms':  [],
    'det_box_area_mean': [],

    # Tracking metrics
    'track_count':     [],
    'track_ids':       [],

    # Pose metrics
    'pose_count':      [],
    'pose_conf_mean':  [],
    'pose_kpt_visible': [],   # mean visible keypoints per person

    # Pipeline throughput
    'pipeline_fps':    [],
    'stage_fps_det':   [],
    'stage_fps_trk':   [],
    'stage_fps_pose':  [],
}

PRINT_EVERY = max(1, len(frames) // 10)
pipeline_t0 = time.perf_counter()
prev_t = pipeline_t0
total_persons_seen = set()

for idx, (frame, frame_idx) in enumerate(frames):
    ts = frame_idx / VIDEO_FPS

    # ── Detection ─────────────────────────────────────────────────────
    t0 = time.perf_counter()
    dets = human_detector.detect_frame(frame, frame_idx)
    det_elapsed = time.perf_counter() - t0

    det_confs   = [d.confidence for d in dets] if dets else [0.0]
    det_areas   = [(d.x2-d.x1)*(d.y2-d.y1) for d in dets] if dets else [0.0]

    # ── Tracking ──────────────────────────────────────────────────────
    t1 = time.perf_counter()
    tracks = person_tracker.track_frame(dets, frame_idx, frame=frame)
    trk_elapsed = time.perf_counter() - t1
    for t in tracks:
        total_persons_seen.add(t.track_id)

    # ── Pose estimation ───────────────────────────────────────────────
    t2 = time.perf_counter()
    if tracks:
        poses_dict = pose_estimator.estimate_frame(tracks, frame, frame_idx)
    else:
        poses_dict = {}
    pose_elapsed = time.perf_counter() - t2

    pose_confs, pose_kpt_vis = [], []
    for pose in poses_dict.values():
        kpts = pose.keypoints_array
        conf = kpts[:, 2] if kpts.shape[1] > 2 else np.ones(kpts.shape[0])
        pose_confs.append(float(conf.mean()))
        pose_kpt_vis.append(float((conf > 0.3).sum()))

    # ── Pipeline FPS ─────────────────────────────────────────────────
    now = time.perf_counter()
    frame_time = now - prev_t
    prev_t = now
    fps_now = 1.0 / max(frame_time, 1e-6)

    # ── Record epoch metrics ─────────────────────────────────────────
    epoch_metrics['frame_idx'].append(frame_idx)
    epoch_metrics['timestamp_sec'].append(round(ts, 3))

    epoch_metrics['det_count'].append(len(dets))
    epoch_metrics['det_conf_mean'].append(round(float(np.mean(det_confs)), 4))
    epoch_metrics['det_conf_max'].append(round(float(np.max(det_confs)),  4))
    epoch_metrics['det_conf_min'].append(round(float(np.min(det_confs)),  4))
    epoch_metrics['det_latency_ms'].append(round(det_elapsed * 1000, 2))
    epoch_metrics['det_box_area_mean'].append(round(float(np.mean(det_areas)), 1))

    epoch_metrics['track_count'].append(len(tracks))
    epoch_metrics['track_ids'].append([t.track_id for t in tracks])

    epoch_metrics['pose_count'].append(len(poses_dict))
    epoch_metrics['pose_conf_mean'].append(round(float(np.mean(pose_confs)) if pose_confs else 0.0, 4))
    epoch_metrics['pose_kpt_visible'].append(round(float(np.mean(pose_kpt_vis)) if pose_kpt_vis else 0.0, 2))

    epoch_metrics['pipeline_fps'].append(round(fps_now, 2))
    epoch_metrics['stage_fps_det'].append(round(1.0/max(det_elapsed,1e-6),  2))
    epoch_metrics['stage_fps_trk'].append(round(1.0/max(trk_elapsed,1e-6),  2))
    epoch_metrics['stage_fps_pose'].append(round(1.0/max(pose_elapsed,1e-6), 2))

    if (idx + 1) % PRINT_EVERY == 0 or idx == 0:
        elapsed = now - pipeline_t0
        est_total = elapsed / (idx+1) * len(frames)
        pct = (idx+1)/len(frames)*100
        print(f'  Epoch {idx+1:>4}/{len(frames)} ({pct:5.1f}%) | '
              f'Det: {len(dets)} ({np.mean(det_confs):.2f} conf) | '
              f'Tracks: {len(tracks)} | '
              f'Poses: {len(poses_dict)} | '
              f'FPS: {fps_now:.1f} | '
              f'ETA: {max(0, est_total-elapsed):.0f}s')

total_elapsed = time.perf_counter() - pipeline_t0
print(f'\n✅ Inference complete')
print(f'   Total frames processed : {len(frames)}')
print(f'   Total time             : {total_elapsed:.1f}s')
print(f'   Average pipeline FPS   : {len(frames)/total_elapsed:.2f}')
print(f'   Unique persons tracked : {len(total_persons_seen)}')

Running inference... (this is the evaluation loop — each frame = 1 epoch step)

WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
  Epoch    1/354 (  0.3%) | Det: 2 (0.87 conf) | Tracks: 0 | Poses: 0 | FPS: 0.1 | ETA: 2726s
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


2026-07-04 09:25:18 | WARNING  | pipeline.tracker:track_frame:69 | Tracking failed at frame 1: ByteTrackTracker._kalman_gain_fn() takes 4 positional arguments but 5 were given


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


2026-07-04 09:25:19 | WARNING  | pipeline.tracker:track_frame:69 | Tracking failed at frame 2: ByteTrackTracker._kalman_gain_fn() takes 4 positional arguments but 5 were given


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


...

2026-07-04 09:25:39 | WARNING  | pipeline.tracker:track_frame:69 | Tracking failed at frame 77: ByteTrackTracker._kalman_gain_fn() takes 4 positional arguments but 5 were given


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


2026-07-04 09:25:39 | WARNING  | pipeline.tracker:track_frame:69 | Tracking failed at frame 78: ByteTrackTracker._kalman_gain_fn() takes 4 positional arguments but 5 were given


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'cvtColor'
> Overload resolution failed:
>  - src is not a numpy array, neither a scalar
>  - Expected Ptr<cv::UMat> for argument 'src'


## 4. Epoch Metrics Summary

In [9]:
# ── Safety fallback (in case inference cell was interrupted) ──────────────────
import time as _time
if 'total_elapsed' not in dir():
    total_elapsed = sum(epoch_metrics['det_latency_ms']) / 1000.0 if epoch_metrics['det_latency_ms'] else 0.0
if 'total_persons_seen' not in dir():
    total_persons_seen = set()
    for tids in epoch_metrics['track_ids']:
        total_persons_seen.update(tids)
if 'pipeline_t0' not in dir():
    pipeline_t0 = _time.perf_counter()

# ── Summary statistics ────────────────────────────────────────────────────────
N = len(epoch_metrics['frame_idx'])
frames_with_det  = sum(1 for c in epoch_metrics['det_count'] if c > 0)
frames_with_pose = sum(1 for c in epoch_metrics['pose_count'] if c > 0)
det_recall  = frames_with_det  / N * 100
pose_recall = frames_with_pose / N * 100

conf_vals  = [c for c in epoch_metrics['det_conf_mean']  if c > 0]
pose_vals  = [c for c in epoch_metrics['pose_conf_mean'] if c > 0]
kpt_vals   = [k for k in epoch_metrics['pose_kpt_visible'] if k > 0]

summary = {
    'Total Epochs (Frames)':        N,
    'Video Duration (s)':           round(VIDEO_DUR, 2),
    'Total Inference Time (s)':     round(total_elapsed, 2),
    'Mean Pipeline FPS':            round(float(np.mean(epoch_metrics['pipeline_fps'])), 2),
    'Peak Pipeline FPS':            round(float(np.max(epoch_metrics['pipeline_fps'])),  2),
    'Detection Recall (%)':         round(det_recall,  1),
    'Mean Det Confidence':          round(float(np.mean(conf_vals)) if conf_vals else 0.0, 4),
    'Mean Det Latency (ms)':        round(float(np.mean(epoch_metrics['det_latency_ms'])), 2),
    'Pose Estimation Recall (%)':   round(pose_recall, 1),
    'Mean Pose Confidence':         round(float(np.mean(pose_vals)) if pose_vals else 0.0, 4),
    'Mean Visible Keypoints':       round(float(np.mean(kpt_vals))  if kpt_vals  else 0.0, 2),
    'Unique Persons Tracked':       len(total_persons_seen),
    'Speedup vs Real-time (×)':     round(float(np.mean(epoch_metrics['pipeline_fps'])) / VIDEO_FPS, 2),
}

print('=' * 55)
print('  HumanMM Pipeline — Epoch Metrics Summary')
print('=' * 55)
for k, v in summary.items():
    print(f'  {k:<35}: {v}')
print('=' * 55)

# Save summary JSON
import json
Path(f'{OUTPUT_DIR}/eval_figures').mkdir(parents=True, exist_ok=True)
with open(f'{OUTPUT_DIR}/eval_figures/epoch_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\n✅ Summary saved → {OUTPUT_DIR}/eval_figures/epoch_summary.json')

  HumanMM Pipeline — Epoch Metrics Summary
  Total Epochs (Frames)              : 79
  Video Duration (s)                 : 11.8
  Total Inference Time (s)           : 27.85
  Mean Pipeline FPS                  : 3.91
  Peak Pipeline FPS                  : 7.29
  Detection Recall (%)               : 100.0
  Mean Det Confidence                : 0.8438
  Mean Det Latency (ms)              : 352.55
  Pose Estimation Recall (%)         : 0.0
  Mean Pose Confidence               : 0.0
  Mean Visible Keypoints             : 0.0
  Unique Persons Tracked             : 2
  Speedup vs Real-time (×)           : 0.13

✅ Summary saved → outputs/eval_figures/epoch_summary.json


## 5. Epoch Plots — Detection Metrics

In [10]:
# ── Smooth helper ─────────────────────────────────────────────────────────────
def smooth(y, w=7):
    if len(y) < w:
        return np.array(y, dtype=float)
    arr = np.array(y, dtype=float)
    kernel = np.ones(w) / w
    return np.convolve(arr, kernel, mode='same')

epochs   = np.array(epoch_metrics['frame_idx'])
times    = np.array(epoch_metrics['timestamp_sec'])

det_count  = np.array(epoch_metrics['det_count'])
det_conf   = np.array(epoch_metrics['det_conf_mean'])
det_conf_h = np.array(epoch_metrics['det_conf_max'])
det_conf_l = np.array(epoch_metrics['det_conf_min'])
det_lat    = np.array(epoch_metrics['det_latency_ms'])
box_area   = np.array(epoch_metrics['det_box_area_mean'])

# ── Figure 1: Detection Confidence & Count over Epochs ───────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Detection Metrics per Epoch (Frame)', fontsize=14, color='#e6edf3', y=0.98)

ax = axes[0]
ax.fill_between(epochs, det_conf_l, det_conf_h, color=ACCENT_GREEN, alpha=0.15, label='Conf range')
ax.plot(epochs, det_conf,        color='#444c56', linewidth=0.8, alpha=0.6)
ax.plot(epochs, smooth(det_conf, 15), color=ACCENT_GREEN, linewidth=2.0, label='Mean conf (smoothed)')
ax.axhline(0.5, color=ACCENT_YELLOW, linewidth=1.0, linestyle='--', label='0.50 threshold')
ax.set_ylabel('Detection Confidence')
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True)
ax.set_title('Per-Epoch Detection Confidence', loc='left', fontsize=11)

ax = axes[1]
ax.bar(epochs, det_count, width=1.0, color=ACCENT_BLUE, alpha=0.7, label='Persons detected')
ax.plot(epochs, smooth(det_count, 15), color='white', linewidth=1.5, label='Trend')
ax.set_ylabel('Persons Detected')
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(loc='upper right', fontsize=9)
ax.grid(True)
ax.set_title('Per-Epoch Detection Count', loc='left', fontsize=11)

ax = axes[2]
ax.plot(epochs, det_lat,        color='#444c56', linewidth=0.8, alpha=0.6)
ax.plot(epochs, smooth(det_lat, 15), color=ACCENT_ORANGE, linewidth=2.0, label='Latency (ms)')
ax.axhline(float(np.mean(det_lat)), color=ACCENT_RED, linewidth=1.0, linestyle='--',
           label=f'Mean={np.mean(det_lat):.1f}ms')
ax.set_xlabel('Epoch (Frame Index)')
ax.set_ylabel('Det Latency (ms)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True)
ax.set_title('Per-Epoch Inference Latency', loc='left', fontsize=11)

plt.tight_layout()
path = FIG_DIR / 'fig1_detection_epochs.png'
plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'✅ Saved → {path}')

✅ Saved → outputs\eval_figures\fig1_detection_epochs.png


## 6. Epoch Plots — Tracking Metrics

In [11]:
track_count = np.array(epoch_metrics['track_count'])
pose_count  = np.array(epoch_metrics['pose_count'])
pose_conf   = np.array(epoch_metrics['pose_conf_mean'])
pose_kv     = np.array(epoch_metrics['pose_kpt_visible'])

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle('Tracking & Pose Metrics per Epoch', fontsize=14, color='#e6edf3', y=0.99)

ax = axes[0]
ax.fill_between(epochs, 0, track_count, color=ACCENT_PURPLE, alpha=0.25)
ax.plot(epochs, track_count, color=ACCENT_PURPLE, linewidth=1.2, label='Active tracks')
ax.plot(epochs, det_count,   color=ACCENT_BLUE,   linewidth=1.0, linestyle='--', alpha=0.7, label='Detections')
ax.set_ylabel('Count')
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(loc='upper right', fontsize=9)
ax.grid(True)
ax.set_title('Active Tracks vs Raw Detections per Epoch', loc='left', fontsize=11)

ax = axes[1]
ax.fill_between(epochs, 0, pose_conf, color=ACCENT_ORANGE, alpha=0.20)
ax.plot(epochs, pose_conf, color='#444c56', linewidth=0.8, alpha=0.5)
ax.plot(epochs, smooth(pose_conf, 15), color=ACCENT_ORANGE, linewidth=2.0, label='Pose confidence')
ax2 = ax.twinx()
ax2.plot(epochs, pose_kv, color=ACCENT_YELLOW, linewidth=1.2, alpha=0.6, linestyle=':', label='Visible kpts')
ax2.set_ylabel('Visible Keypoints', color=ACCENT_YELLOW)
ax2.tick_params(axis='y', labelcolor=ACCENT_YELLOW)
ax.set_xlabel('Epoch (Frame Index)')
ax.set_ylabel('Pose Confidence')
ax.set_ylim(0, 1.05)
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labs1+labs2, loc='upper right', fontsize=9)
ax.grid(True)
ax.set_title('Per-Epoch Pose Confidence & Visible Keypoints', loc='left', fontsize=11)

plt.tight_layout()
path = FIG_DIR / 'fig2_tracking_pose_epochs.png'
plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'✅ Saved → {path}')

✅ Saved → outputs\eval_figures\fig2_tracking_pose_epochs.png


## 7. Pipeline FPS per Epoch

In [12]:
fps_pipe = np.array(epoch_metrics['pipeline_fps'])
fps_det  = np.array(epoch_metrics['stage_fps_det'])
fps_trk  = np.array(epoch_metrics['stage_fps_trk'])
fps_pose = np.array(epoch_metrics['stage_fps_pose'])

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Pipeline Throughput per Epoch', fontsize=14, color='#e6edf3')

ax = axes[0]
ax.plot(epochs, fps_pipe,            color='#444c56', linewidth=0.6, alpha=0.4)
ax.plot(epochs, smooth(fps_pipe, 15), color=ACCENT_GREEN, linewidth=2.0, label='Pipeline FPS')
ax.axhline(float(np.mean(fps_pipe)),  color=ACCENT_YELLOW, linewidth=1.2, linestyle='--',
           label=f'Mean = {np.mean(fps_pipe):.1f} fps')
ax.axhline(VIDEO_FPS, color=ACCENT_RED, linewidth=1.0, linestyle=':', label=f'Real-time ({VIDEO_FPS:.0f} fps)')
ax.set_ylabel('FPS')
ax.set_title('End-to-End Pipeline FPS per Epoch', loc='left', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True)

ax = axes[1]
ax.plot(epochs, smooth(fps_det,  9), color=ACCENT_BLUE,   linewidth=1.8, label='Detection FPS')
ax.plot(epochs, smooth(fps_trk,  9), color=ACCENT_GREEN,  linewidth=1.8, label='Tracking FPS')
ax.plot(epochs, smooth(fps_pose, 9), color=ACCENT_ORANGE, linewidth=1.8, label='Pose FPS')
ax.set_xlabel('Epoch (Frame Index)')
ax.set_ylabel('Stage FPS')
ax.set_title('Per-Stage FPS Breakdown per Epoch', loc='left', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True)

plt.tight_layout()
path = FIG_DIR / 'fig3_fps_epochs.png'
plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'✅ Saved → {path}')

✅ Saved → outputs\eval_figures\fig3_fps_epochs.png


## 8. Distribution Histograms

In [13]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('HumanMM — Metric Distributions across All Epochs', fontsize=14, color='#e6edf3')

def hist(ax, data, label, color, xlabel, bins=30):
    data_clean = [d for d in data if d > 0]
    if not data_clean:
        ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
        return
    ax.hist(data_clean, bins=bins, color=color, alpha=0.8, edgecolor='none')
    mean_v = np.mean(data_clean)
    ax.axvline(mean_v, color='white', linewidth=1.5, linestyle='--', label=f'Mean={mean_v:.3f}')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel('Epoch Count', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, axis='y')

hist(axes[0,0], epoch_metrics['det_conf_mean'],  'Detection Confidence',       ACCENT_GREEN,  'Confidence Score')
hist(axes[0,1], epoch_metrics['det_count'],       'Persons Detected per Epoch', ACCENT_BLUE,   'Person Count', bins=max(1,int(max(det_count))+1))
hist(axes[0,2], epoch_metrics['det_latency_ms'],  'Detection Latency',          ACCENT_ORANGE, 'Latency (ms)')
hist(axes[1,0], epoch_metrics['pose_conf_mean'],  'Pose Confidence',            ACCENT_PURPLE, 'Confidence Score')
hist(axes[1,1], epoch_metrics['pose_kpt_visible'],'Visible Keypoints / Person', ACCENT_YELLOW, 'Keypoint Count')
hist(axes[1,2], epoch_metrics['pipeline_fps'],    'Pipeline FPS',               ACCENT_RED,    'FPS')

plt.tight_layout()
path = FIG_DIR / 'fig4_distributions.png'
plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'✅ Saved → {path}')

✅ Saved → outputs\eval_figures\fig4_distributions.png


## 9. Pipeline Stage Comparison — Bar Chart

In [ ]:
# Compute mean latency per stage
lat_det  = float(np.mean(epoch_metrics['det_latency_ms']))
lat_trk  = float(np.mean([1000/f for f in epoch_metrics['stage_fps_trk']  if f > 0]))
lat_pose = float(np.mean([1000/f for f in epoch_metrics['stage_fps_pose'] if f > 0]))

stages  = ['Detection\n(YOLO11)', 'Tracking\n(ByteTrack)', 'Pose Estimation\n(MediaPipe)']
lats    = [lat_det, lat_trk, lat_pose]
colors  = [ACCENT_BLUE, ACCENT_GREEN, ACCENT_ORANGE]

fps_per_stage = [1000/l for l in lats]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Stage-wise Performance Breakdown', fontsize=14, color='#e6edf3')

bars = axes[0].bar(stages, lats, color=colors, width=0.5, edgecolor='none')
for bar, val in zip(bars, lats):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{val:.1f}ms', ha='center', va='bottom', fontsize=10, color='white')
axes[0].set_ylabel('Mean Latency (ms)')
axes[0].set_title('Mean Stage Latency', fontsize=11)
axes[0].grid(True, axis='y')

bars2 = axes[1].bar(stages, fps_per_stage, color=colors, width=0.5, edgecolor='none')
for bar, val in zip(bars2, fps_per_stage):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=10, color='white')
axes[1].axhline(VIDEO_FPS, color=ACCENT_RED, linewidth=1.2, linestyle='--',
                label=f'Real-time target ({VIDEO_FPS:.0f} fps)')
axes[1].set_ylabel('Throughput (FPS)')
axes[1].set_title('Stage Throughput vs Real-time', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, axis='y')

plt.tight_layout()
path = FIG_DIR / 'fig5_stage_comparison.png'
plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'✅ Saved → {path}')

## 10. Confidence Threshold Sensitivity Analysis

In [14]:
# Simulate how many detections survive at different thresholds
all_confs = []
det_results = human_detector.get_results()
for dets in det_results.values():
    for d in dets:
        all_confs.append(d.confidence)

thresholds = np.arange(0.10, 0.95, 0.025)
survival   = [sum(c >= t for c in all_confs) / max(1, len(all_confs)) * 100 for t in thresholds]
frame_recall = []
for t in thresholds:
    frames_hit = sum(
        1 for dets in det_results.values()
        if any(d.confidence >= t for d in dets)
    )
    frame_recall.append(frames_hit / N * 100)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, survival, color=ACCENT_GREEN, linewidth=2.2, label='Detection survival rate (%)')
ax.plot(thresholds, frame_recall, color=ACCENT_BLUE, linewidth=2.2, linestyle='--', label='Frame recall (%)')
ax.axvline(0.40, color=ACCENT_YELLOW, linewidth=1.2, linestyle=':', label='Current threshold (0.40)')
ax.fill_between(thresholds, survival, alpha=0.08, color=ACCENT_GREEN)
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('Percentage (%)')
ax.set_title('Confidence Threshold Sensitivity — Impact on Detection Recall', fontsize=12)
ax.set_xlim(0.10, 0.90)
ax.set_ylim(0, 105)
ax.legend(fontsize=10)
ax.grid(True)

plt.tight_layout()
path = FIG_DIR / 'fig6_threshold_sensitivity.png'
plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'✅ Saved → {path}')

✅ Saved → outputs\eval_figures\fig6_threshold_sensitivity.png


## 11. Person Track ID Timeline

In [15]:
# Build per-track active frame timeline
track_timeline = {}  # track_id -> list of frame_indices where active
for fidx, tids in zip(epoch_metrics['frame_idx'], epoch_metrics['track_ids']):
    for tid in tids:
        track_timeline.setdefault(tid, []).append(fidx)

sorted_ids = sorted(track_timeline.keys())

if sorted_ids:
    import colorsys
    def id_color(tid):
        golden = 0.618033988749895
        hue = (tid * golden) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, 0.85, 1.0)
        return (r, g, b)

    fig, ax = plt.subplots(figsize=(14, max(3, len(sorted_ids) * 0.6 + 1)))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#161b22')

    for row, tid in enumerate(sorted_ids):
        active_frames = sorted(track_timeline[tid])
        color = id_color(tid)
        # Draw segments
        start = active_frames[0]
        for i in range(1, len(active_frames)):
            if active_frames[i] - active_frames[i-1] > 5:  # gap > 5 frames
                ax.barh(row, active_frames[i-1] - start + 1, left=start,
                        height=0.65, color=color, alpha=0.85)
                start = active_frames[i]
        ax.barh(row, active_frames[-1] - start + 1, left=start,
                height=0.65, color=color, alpha=0.85)
        ax.text(-5, row, f'#{tid}', ha='right', va='center', fontsize=8.5,
                color=color, fontweight='bold')

    ax.set_xlim(-1, VIDEO_FRAMES + 1)
    ax.set_ylim(-0.5, len(sorted_ids) - 0.5)
    ax.set_yticks([])
    ax.set_xlabel('Frame Index (Epoch)')
    ax.set_title(f'Track ID Timeline — {len(sorted_ids)} Unique Persons', fontsize=12)
    ax.grid(True, axis='x', alpha=0.3)
    ax.spines['left'].set_visible(False)

    plt.tight_layout()
    path = FIG_DIR / 'fig7_track_timeline.png'
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'✅ Saved → {path}')
else:
    print('No tracks to show')

No tracks to show


## 12. Sample Output Frames — Visualisation

In [16]:
# ── Show sample detections at 6 representative epochs ────────────────────────
from visualization.video_renderer import DetectionRenderer

det_viz = DetectionRenderer()

# Pick epochs with at least 1 detection
good_idx = [i for i, c in enumerate(epoch_metrics['det_count']) if c > 0]
sample_positions = np.linspace(0, len(good_idx)-1, 6, dtype=int) if len(good_idx) >= 6 else list(range(len(good_idx)))
sample_epochs = [good_idx[p] for p in sample_positions]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
fig.suptitle('Sample Detection Outputs — Representative Epochs', fontsize=13, color='#e6edf3')
axes = axes.flatten()

det_results_all = human_detector.get_results()

for plot_i, epoch_i in enumerate(sample_epochs):
    fidx = epoch_metrics['frame_idx'][epoch_i]
    frame_bgr = frames[epoch_i][0].copy()
    dets_here  = det_results_all.get(fidx, [])
    fps_here   = epoch_metrics['pipeline_fps'][epoch_i]
    lat_here   = epoch_metrics['det_latency_ms'][epoch_i]

    annotated = det_viz.render(
        frame_bgr, dets_here,
        frame_idx=fidx, fps=fps_here, latency_ms=lat_here
    )

    rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    rgb_small = cv2.resize(rgb, (640, 360))
    axes[plot_i].imshow(rgb_small)
    axes[plot_i].set_title(
        f'Epoch {fidx} | {len(dets_here)} det | {fps_here:.1f}fps | {lat_here:.0f}ms',
        fontsize=8.5
    )
    axes[plot_i].axis('off')

# Hide unused
for i in range(len(sample_epochs), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
path = FIG_DIR / 'fig8_sample_detections.png'
plt.savefig(path, dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'✅ Saved → {path}')

✅ Saved → outputs\eval_figures\fig8_sample_detections.png


## 13. Load Existing Stage Videos — Thumbnail Summary

In [17]:
# ── If stage videos exist from a previous run, extract thumbnails ─────────────
stage_videos = {
    'Detection (02)':       f'{OUTPUT_DIR}/02_detection.mp4',
    'Tracking (03)':        f'{OUTPUT_DIR}/03_tracking.mp4',
    'Pose Estimation (04)': f'{OUTPUT_DIR}/04_pose.mp4',
    '3D Mesh (05)':         f'{OUTPUT_DIR}/05_mesh.mp4',
    'Trajectory (06)':      f'{OUTPUT_DIR}/06_trajectory.mp4',
    'Final Demo (07)':      f'{OUTPUT_DIR}/07_final.mp4',
}

existing = {k: v for k, v in stage_videos.items() if Path(v).exists()}
print(f'Found {len(existing)}/{len(stage_videos)} stage videos\n')

if existing:
    cols = min(3, len(existing))
    rows = (len(existing) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5.5 * cols, 3.8 * rows))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = axes.reshape(1, -1)
    fig.suptitle('Stage Video Thumbnails — Pipeline Outputs', fontsize=13, color='#e6edf3')

    for i, (label, vpath) in enumerate(existing.items()):
        r, c = divmod(i, cols)
        cap = cv2.VideoCapture(vpath)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)  # mid-point frame
        ret, thumb = cap.read()
        cap.release()
        ax = axes[r, c]
        if ret:
            rgb = cv2.cvtColor(thumb, cv2.COLOR_BGR2RGB)
            ax.imshow(rgb)
            ax.set_title(label, fontsize=9.5, pad=5)
        else:
            ax.text(0.5, 0.5, 'Failed', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

    for i in range(len(existing), rows * cols):
        r, c = divmod(i, cols)
        axes[r, c].set_visible(False)

    plt.tight_layout()
    path = FIG_DIR / 'fig9_stage_thumbnails.png'
    plt.savefig(path, dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'✅ Saved → {path}')

Found 6/6 stage videos

✅ Saved → outputs\eval_figures\fig9_stage_thumbnails.png


## 14. Combined Dashboard — All Metrics in One View

In [19]:
# ── Safety: recompute any variables that depend on earlier cells ──────────────
import colorsys as _cs

# Stage latency vars (from Cell 13 — recompute if missing)
try:
    stages; lats; colors
except NameError:
    lat_det  = float(np.mean(epoch_metrics['det_latency_ms']))
    lat_trk  = float(np.mean([1000/f for f in epoch_metrics['stage_fps_trk']  if 0 < f < 1e5]))
    lat_pose = float(np.mean([1000/f for f in epoch_metrics['stage_fps_pose'] if 0 < f < 1e5]))
    stages   = ['Detection\n(YOLO11)', 'Tracking\n(ByteTrack)', 'Pose\n(MediaPipe)']
    lats     = [lat_det, lat_trk, lat_pose]
    colors   = [ACCENT_BLUE, ACCENT_GREEN, ACCENT_ORANGE]

# Threshold sensitivity vars (from Cell 14 — recompute if missing)
try:
    thresholds; survival; frame_recall
except NameError:
    all_confs = []
    for dets_list in human_detector.get_results().values():
        for d in dets_list:
            all_confs.append(d.confidence)
    thresholds = np.arange(0.10, 0.95, 0.025)
    total_det_results = human_detector.get_results()
    survival = [
        sum(c >= t for c in all_confs) / max(1, len(all_confs)) * 100
        for t in thresholds
    ]
    frame_recall = [
        sum(1 for dets in total_det_results.values() if any(d.confidence >= t for d in dets))
        / N * 100
        for t in thresholds
    ]

# Epoch arrays (from Cell 9)
try:
    epochs; det_conf; det_conf_l; det_conf_h; det_lat; track_count; fps_pipe
except NameError:
    epochs      = np.array(epoch_metrics['frame_idx'])
    det_conf    = np.array(epoch_metrics['det_conf_mean'])
    det_conf_h  = np.array(epoch_metrics['det_conf_max'])
    det_conf_l  = np.array(epoch_metrics['det_conf_min'])
    det_lat     = np.array(epoch_metrics['det_latency_ms'])
    track_count = np.array(epoch_metrics['track_count'])
    fps_pipe    = np.array(epoch_metrics['pipeline_fps'])

# ── Dashboard ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.suptitle('HumanMM — Complete Evaluation Dashboard', fontsize=16,
             color='#e6edf3', fontweight='bold', y=0.99)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# Row 0: Detection confidence full width
ax0 = fig.add_subplot(gs[0, :])
ax0.fill_between(epochs, det_conf_l, det_conf_h, color=ACCENT_GREEN, alpha=0.12)
ax0.plot(epochs, det_conf,             color='#444c56', linewidth=0.7, alpha=0.4)
ax0.plot(epochs, smooth(det_conf, 15), color=ACCENT_GREEN, linewidth=2.0, label='Mean confidence')
ax0.fill_between(epochs, 0, track_count.astype(float) / max(track_count.max(), 1),
                 color=ACCENT_PURPLE, alpha=0.15, label='Track count (norm)')
ax0.axhline(0.50, color=ACCENT_YELLOW, linewidth=1.0, linestyle='--', label='0.50 threshold')
ax0.set_ylabel('Confidence / Norm. Count')
ax0.set_title('Detection Confidence & Track Density per Epoch', loc='left')
ax0.set_xlim(epochs[0], epochs[-1])
ax0.set_ylim(0, 1.1)
ax0.legend(fontsize=8, ncol=3)
ax0.grid(True)
ax0.set_xlabel('Epoch (Frame)')

# Row 1, col 0: Latency histogram
ax1 = fig.add_subplot(gs[1, 0])
ax1.hist(epoch_metrics['det_latency_ms'], bins=25, color=ACCENT_ORANGE, alpha=0.85, edgecolor='none')
ax1.axvline(float(np.mean(det_lat)), color='white', linestyle='--', linewidth=1.2,
            label=f'Mean={np.mean(det_lat):.0f}ms')
ax1.set_title('Detection Latency Dist.', fontsize=10)
ax1.set_xlabel('ms'); ax1.set_ylabel('Epochs'); ax1.legend(fontsize=8); ax1.grid(True, axis='y')

# Row 1, col 1: Confidence histogram
ax2 = fig.add_subplot(gs[1, 1])
clean_conf = [c for c in epoch_metrics['det_conf_mean'] if c > 0]
if clean_conf:
    ax2.hist(clean_conf, bins=25, color=ACCENT_GREEN, alpha=0.85, edgecolor='none')
    ax2.axvline(np.mean(clean_conf), color='white', linestyle='--', linewidth=1.2,
                label=f'Mean={np.mean(clean_conf):.3f}')
ax2.set_title('Detection Confidence Dist.', fontsize=10)
ax2.set_xlabel('Confidence'); ax2.set_ylabel('Epochs'); ax2.legend(fontsize=8); ax2.grid(True, axis='y')

# Row 1, col 2: Person count distribution
ax3 = fig.add_subplot(gs[1, 2])
unique_counts, cnts = np.unique(epoch_metrics['det_count'], return_counts=True)
ax3.bar(unique_counts, cnts, color=ACCENT_BLUE, alpha=0.85, width=0.5, edgecolor='none')
ax3.set_title('Persons per Epoch Dist.', fontsize=10)
ax3.set_xlabel('Person Count'); ax3.set_ylabel('Epochs')
ax3.xaxis.set_major_locator(MaxNLocator(integer=True)); ax3.grid(True, axis='y')

# Row 1, col 3: FPS histogram
ax4 = fig.add_subplot(gs[1, 3])
ax4.hist(epoch_metrics['pipeline_fps'], bins=25, color=ACCENT_RED, alpha=0.85, edgecolor='none')
ax4.axvline(float(np.mean(fps_pipe)), color='white', linestyle='--', linewidth=1.2,
            label=f'Mean={np.mean(fps_pipe):.1f}')
ax4.axvline(VIDEO_FPS, color=ACCENT_YELLOW, linestyle=':', linewidth=1.2,
            label=f'Real-time={VIDEO_FPS:.0f}')
ax4.set_title('Pipeline FPS Dist.', fontsize=10)
ax4.set_xlabel('FPS'); ax4.set_ylabel('Epochs'); ax4.legend(fontsize=7); ax4.grid(True, axis='y')

# Row 2: Stage latency bars
ax5 = fig.add_subplot(gs[2, :2])
bars = ax5.bar(stages, lats, color=colors, width=0.45, edgecolor='none')
for bar, val in zip(bars, lats):
    ax5.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{val:.1f}ms', ha='center', va='bottom', fontsize=10, color='white')
ax5.set_title('Stage Latency Comparison (ms)', fontsize=10)
ax5.set_ylabel('Mean Latency (ms)'); ax5.grid(True, axis='y')

# Row 2: Threshold sensitivity
ax6 = fig.add_subplot(gs[2, 2:])
ax6.plot(thresholds, survival,     color=ACCENT_GREEN, linewidth=2.0, label='Detection rate %')
ax6.plot(thresholds, frame_recall, color=ACCENT_BLUE,  linewidth=2.0, linestyle='--', label='Frame recall %')
ax6.axvline(0.40, color=ACCENT_YELLOW, linewidth=1.2, linestyle=':', label='Current thresh')
ax6.set_title('Confidence Threshold Sensitivity', fontsize=10)
ax6.set_xlabel('Threshold'); ax6.set_ylabel('%'); ax6.legend(fontsize=8); ax6.grid(True)
ax6.set_ylim(0, 105)

path = FIG_DIR / 'fig10_dashboard.png'
plt.savefig(path, dpi=130, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'✅ Dashboard saved → {path}')

✅ Dashboard saved → outputs\eval_figures\fig10_dashboard.png


## 15. Save Epoch Metrics to CSV & JSON

In [20]:
import csv

# Save detailed epoch metrics CSV
csv_cols = ['frame_idx','timestamp_sec','det_count','det_conf_mean','det_conf_max',
            'det_conf_min','det_latency_ms','det_box_area_mean','track_count',
            'pose_count','pose_conf_mean','pose_kpt_visible','pipeline_fps']

csv_path = FIG_DIR / 'epoch_metrics.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=csv_cols)
    writer.writeheader()
    for i in range(N):
        writer.writerow({k: epoch_metrics[k][i] for k in csv_cols})

print(f'✅ Epoch metrics CSV → {csv_path}')
print(f'   Rows: {N}  Columns: {len(csv_cols)}')
print()

# Also save all figures list
figs = sorted(FIG_DIR.glob('*.png'))
print(f'📊 All saved figures ({len(figs)} total):')
for f in figs:
    sz = f.stat().st_size // 1024
    print(f'   {f.name:<45} {sz:>5} KB')

✅ Epoch metrics CSV → outputs\eval_figures\epoch_metrics.csv
   Rows: 79  Columns: 13

📊 All saved figures (8 total):
   fig10_dashboard.png                             253 KB
   fig1_detection_epochs.png                       186 KB
   fig2_tracking_pose_epochs.png                   101 KB
   fig3_fps_epochs.png                             135 KB
   fig4_distributions.png                          133 KB
   fig6_threshold_sensitivity.png                   71 KB
   fig8_sample_detections.png                     1382 KB
   fig9_stage_thumbnails.png                      1021 KB


## 16. Final Report Card

In [21]:
print('=' * 62)
print('  HumanMM — Final Evaluation Report Card')
print('  Global Human Motion Recovery from Multi-Shot Videos')
print('=' * 62)
print()
print(f'  📹 Input Video      : {Path(VIDEO_PATH).name}')
print(f'  🎞️  Total Epochs     : {N} frames processed')
print(f'  ⏱️  Total Time       : {total_elapsed:.1f}s')
print()
print('  STAGE RESULTS')
print('  ┌─────────────────────────┬──────────┬───────────────┐')
print('  │ Stage                   │ FPS      │ Key Metric    │')
print('  ├─────────────────────────┼──────────┼───────────────┤')
mean_fps_det  = float(np.mean([f for f in epoch_metrics['stage_fps_det']  if f < 1e5]))
mean_fps_trk  = float(np.mean([f for f in epoch_metrics['stage_fps_trk']  if f < 1e5]))
mean_fps_pose = float(np.mean([f for f in epoch_metrics['stage_fps_pose'] if f < 1e5]))

conf_clean = [c for c in epoch_metrics['det_conf_mean'] if c > 0]
pose_clean = [c for c in epoch_metrics['pose_conf_mean'] if c > 0]

print(f'  │ Human Detection (YOLO11) │ {mean_fps_det:>7.1f}  │ Conf={np.mean(conf_clean):.3f}     │')
print(f'  │ Tracking (ByteTrack)    │ {mean_fps_trk:>7.1f}  │ {len(total_persons_seen)} unique IDs  │')
print(f'  │ Pose (MediaPipe)        │ {mean_fps_pose:>7.1f}  │ Conf={np.mean(pose_clean) if pose_clean else 0:.3f}     │')
print(f'  │ END-TO-END              │ {len(frames)/total_elapsed:>7.2f}  │ {det_recall:.1f}% recall   │')
print('  └─────────────────────────┴──────────┴───────────────┘')
print()
print(f'  Figures saved to : {FIG_DIR}')
print(f'  CSV saved to     : {FIG_DIR}/epoch_metrics.csv')
print()
print('  STATUS: ✅ Evaluation Complete')
print('=' * 62)

  HumanMM — Final Evaluation Report Card
  Global Human Motion Recovery from Multi-Shot Videos

  📹 Input Video      : demo.mp4
  🎞️  Total Epochs     : 79 frames processed
  ⏱️  Total Time       : 27.9s

  STAGE RESULTS
  ┌─────────────────────────┬──────────┬───────────────┐
  │ Stage                   │ FPS      │ Key Metric    │
  ├─────────────────────────┼──────────┼───────────────┤
  │ Human Detection (YOLO11) │     4.1  │ Conf=0.844     │
  │ Tracking (ByteTrack)    │   130.3  │ 2 unique IDs  │
  │ Pose (MediaPipe)        │     nan  │ Conf=0.000     │
  │ END-TO-END              │   12.71  │ 100.0% recall   │
  └─────────────────────────┴──────────┴───────────────┘

  Figures saved to : outputs\eval_figures
  CSV saved to     : outputs\eval_figures/epoch_metrics.csv

  STATUS: ✅ Evaluation Complete
